# Apply Saved Transforms to N-channel IF-prediction Tifs

Variant of `apply_transforms.ipynb` for applying an existing H&E serial-section
registration to **pix2pix IF-prediction tifs**.

The IF tifs are `(H, W, N)` images, downsampled by `SEG_DOWNSAMPLE` (16) from the
original level-0 H&E. Each tif channel is a continuous intensity image (channel 0
= DAPI, etc.).

For each slide:
1. Load the N-channel tif
2. Pad by `PADDING * scale` pixels (mirroring what alignment did at reg level)
3. Resize padded image to match the padded anchor canvas (mirrors `_match_size`)
4. Apply the scaled affine transform
5. Apply the scaled elastic displacement field

**Output** (in `WARPED_OUTPUT_FOLDER`):
```
registered/
├── {stem}.tif           ← warped N-channel tif (all channels)
├── ch0_DAPI/{stem}.tif  ← warped channel 0 (single channel)
├── ch1/{stem}.tif
├── ch2/{stem}.tif
└── ch3/{stem}.tif
```

In [ ]:
import sys, os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tifffile
from pathlib import Path
from PIL import Image

ALIGNMENT_DIR = os.path.dirname(os.path.abspath("__file__"))
if ALIGNMENT_DIR not in sys.path:
    sys.path.insert(0, ALIGNMENT_DIR)

from alignment_pipeline import load_transform, discover_slides
from registration import read_ndpi_level, apply_affine, apply_elastic_transform, get_level_dimensions

## Configuration

In [ ]:
# Folder the alignment was saved to (must contain transforms/affine + transforms/elastic + aligned)
ALIGNMENT_OUTPUT_FOLDER = "/path/to/alignment"

# N-channel IF-prediction tifs to warp
NEW_IMAGE_FOLDER     = "/path/to/images/to/be/registered"
NEW_IMAGE_EXT        = ".tif"
WARPED_OUTPUT_FOLDER = "/output/path"

# The original H&E WSIs the alignment was calculated on (used to recover level-0 / reg-level dims)
ORIGINAL_SLIDE_FOLDER = "/path/to/original/WSIs"
ORIGINAL_SLIDE_EXT    = ".ndpi"
REGISTRATION_LEVEL    = 5   # LEVEL used in align_serial_sections.ipynb

# Padding used during alignment (pixels at registration level) -- matches PADDING in alignment nb
PADDING        = 200

# IF tifs are level-0 / SEG_DOWNSAMPLE (downsized by 16 from original H&E)
SEG_DOWNSAMPLE = 16

# Channel names -> used for the per-channel subfolder names (index i uses CHANNEL_NAMES[i]).
# Channels beyond this list fall back to 'ch{i}'.
CHANNEL_NAMES = ["c1", "c2", "c3", "c4"]

FILL_VALUE   = 0        # IF background = 0
IS_LABEL_MAP = False    # continuous intensities -> linear interpolation
SAVE_COMBINED_TIF = True

## Setup

In [ ]:
def read_image(path: str) -> np.ndarray:
    """Read an N-channel tif as (H, W, C)."""
    ext = Path(path).suffix.lower()
    if ext in (".tif", ".tiff"):
        img = tifffile.imread(path)
        # tifffile may return (C, H, W) for some multi-page tifs -> move channels last
        if img.ndim == 3 and img.shape[0] <= 8 and img.shape[0] < img.shape[-1]:
            img = np.moveaxis(img, 0, -1)
        if img.ndim == 2:
            img = img[:, :, None]
        return img
    else:
        img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        if img is None:
            raise IOError(f"Could not read {path}")
        if img.ndim == 2:
            img = img[:, :, None]
        return img


def resolve_elastic(t: dict, stem: str) -> np.ndarray:
    """Return the elastic displacement field, resolving a stale displacement_file path.

    The saved pkl records a displacement_file pointing at whatever folder alignment
    originally wrote to; if that path has moved, fall back to the elastic/ sibling of
    the affine dir under ALIGNMENT_OUTPUT_FOLDER.
    """
    disp = t.get("elastic_transform")
    if disp is not None:
        return disp
    npy = Path(ALIGNMENT_OUTPUT_FOLDER) / "transforms" / "elastic" / f"{stem}.npy"
    if npy.exists():
        return np.load(str(npy))
    return None


def scale_affine_matrix(M: np.ndarray, sx: float, sy: float) -> np.ndarray:
    """Lift 2x3 affine M from reg frame to seg frame: M_seg = S @ M @ S_inv."""
    M = np.array(M, dtype=np.float64)
    if M.shape == (3, 3):
        M = M[:2, :]
    S     = np.array([[sx, 0, 0], [0, sy, 0], [0, 0, 1]], dtype=np.float64)
    S_inv = np.array([[1/sx, 0, 0], [0, 1/sy, 0], [0, 0, 1]], dtype=np.float64)
    return (S @ np.vstack([M, [0, 0, 1]]) @ S_inv)[:2, :]


def scale_displacement_field(disp: np.ndarray, sx: float, sy: float,
                              out_h: int, out_w: int) -> np.ndarray:
    """Resize displacement field and scale magnitudes."""
    dx = cv2.resize(disp[:, :, 0], (out_w, out_h), interpolation=cv2.INTER_LINEAR) * sx
    dy = cv2.resize(disp[:, :, 1], (out_w, out_h), interpolation=cv2.INTER_LINEAR) * sy
    return np.stack([dx, dy], axis=-1)


def channel_dirname(i: int) -> str:
    name = CHANNEL_NAMES[i] if i < len(CHANNEL_NAMES) else None
    return f"ch{i}_{name}" if name else f"ch{i}"

## Load transforms and compute scale

In [ ]:
affine_dir = Path(ALIGNMENT_OUTPUT_FOLDER) / "transforms" / "affine"
transforms_by_stem = {p.stem: load_transform(str(affine_dir), p.stem)
                      for p in sorted(affine_dir.glob("*.pkl"))}

original_slides = discover_slides(ORIGINAL_SLIDE_FOLDER, ORIGINAL_SLIDE_EXT)
slides_by_stem  = {Path(s).stem: s for s in original_slides}

# Anchor
anchor_stem = next(stem for stem, t in transforms_by_stem.items()
                   if t.get("notes", "").startswith("anchor"))
print(f"Anchor: {anchor_stem}")

# Anchor dims at reg level and level-0
anchor_dims = get_level_dimensions(slides_by_stem[anchor_stem])
W_reg, H_reg = anchor_dims[REGISTRATION_LEVEL]
W0,    H0    = anchor_dims[0]
print(f"Anchor reg-level dims (unpadded): W={W_reg}  H={H_reg}")
print(f"Anchor level-0 dims             : W={W0}  H={H0}")

# Scale from reg-level pixels to seg-level pixels (content only, no padding)
sx = (W0 / SEG_DOWNSAMPLE) / W_reg
sy = (H0 / SEG_DOWNSAMPLE) / H_reg
print(f"Scale reg\u2192seg                   : sx={sx:.4f}  sy={sy:.4f}")

# Padding in seg-level pixels
pad_seg = int(round(PADDING * sx))   # sx \u2248 sy for square pixels
print(f"Padding: {PADDING}px @ reg \u2192 {pad_seg}px @ seg")

# Expected canvas size = unpadded_seg + 2*pad_seg
CANVAS_W = int(round(W0 / SEG_DOWNSAMPLE)) + 2 * pad_seg
CANVAS_H = int(round(H0 / SEG_DOWNSAMPLE)) + 2 * pad_seg
print(f"Padded seg canvas               : W={CANVAS_W}  H={CANVAS_H}")

# Verify against the saved anchor aligned PNG
anchor_png = Path(ALIGNMENT_OUTPUT_FOLDER) / "aligned" / f"{anchor_stem}.png"
_chk = np.array(Image.open(str(anchor_png))).shape[:2]
print(f"Reg canvas from anchor PNG      : H={_chk[0]}  W={_chk[1]}")
CANVAS_W_REG, CANVAS_H_REG = _chk[1], _chk[0]
# True sx/sy mapping padded reg \u2192 padded seg (used to scale M and displacement)
sx_canvas = CANVAS_W / CANVAS_W_REG
sy_canvas = CANVAS_H / CANVAS_H_REG
print(f"Canvas scale (padded/padded)    : sx={sx_canvas:.6f}  sy={sy_canvas:.6f}")

## Apply transforms

Each channel is warped independently (2-D at a time) so `apply_affine` /
`apply_elastic_transform` see a single-channel image, then stacked back into an
N-channel result.

In [ ]:
interp = cv2.INTER_NEAREST if IS_LABEL_MAP else cv2.INTER_LINEAR
out_dir = Path(WARPED_OUTPUT_FOLDER)
out_dir.mkdir(parents=True, exist_ok=True)

all_seg_slides = discover_slides(NEW_IMAGE_FOLDER, NEW_IMAGE_EXT)
seg_by_stem    = {Path(s).stem: s for s in all_seg_slides}
print(f"Found {len(all_seg_slides)} IF tifs\n")


def warp_channel(ch2d: np.ndarray) -> np.ndarray:
    """Pad -> resize -> affine -> elastic a single 2-D channel. Uses module-level t/M/disp_s."""
    ch = cv2.copyMakeBorder(ch2d, pad_seg, pad_seg, pad_seg, pad_seg,
                            cv2.BORDER_CONSTANT, value=FILL_VALUE)
    if ch.shape[:2] != (CANVAS_H, CANVAS_W):
        ch = cv2.resize(ch, (CANVAS_W, CANVAS_H), interpolation=interp)
    if M is not None:
        ch = apply_affine(ch, scale_affine_matrix(M, sx_canvas, sy_canvas),
                          (CANVAS_H, CANVAS_W), border_value=FILL_VALUE, interpolation=interp)
    if disp_s is not None:
        ch = apply_elastic_transform(ch, disp_s, (CANVAS_H, CANVAS_W),
                                     default_value=float(FILL_VALUE), is_label=IS_LABEL_MAP)
    return ch


for stem, t in sorted(transforms_by_stem.items(),
                       key=lambda x: x[1].get("slide_index", 0)):
    seg_path = seg_by_stem.get(stem)
    if seg_path is None:
        print(f"  [{stem}] no IF tif found \u2014 skipping")
        continue

    notes = t.get("notes", "")
    if "skipped" in notes:
        print(f"  [{stem}] skipped slide \u2014 no output")
        continue

    print(f"  [{stem}] processing ...", end=" ", flush=True)

    # 1. Load N-channel tif -> (H, W, C)
    img = read_image(seg_path)
    n_ch = img.shape[2]
    in_dtype = img.dtype

    # Resolve transforms once per slide
    M      = t.get("affine_matrix")
    disp   = resolve_elastic(t, stem)
    disp_s = (scale_displacement_field(disp, sx_canvas, sy_canvas, CANVAS_H, CANVAS_W)
              if disp is not None else None)

    # 2-5. Warp each channel independently, then stack
    warped = np.stack([warp_channel(img[:, :, c]) for c in range(n_ch)], axis=-1)
    warped = warped.astype(in_dtype)

    # Save combined N-channel tif
    if SAVE_COMBINED_TIF:
        tifffile.imwrite(str(out_dir / f"{stem}.tif"), warped)

    # Save per-channel single-channel tifs into subfolders
    for c in range(n_ch):
        cdir = out_dir / channel_dirname(c)
        cdir.mkdir(parents=True, exist_ok=True)
        tifffile.imwrite(str(cdir / f"{stem}.tif"), warped[:, :, c])

    print(f"done \u2192 {warped.shape} {in_dtype}")

print(f"\nAll done.")

## Visualize warped channels

In [ ]:
warped_files = sorted(out_dir.glob("*.tif"))
if not warped_files:
    print("No warped tifs found.")
else:
    sample = tifffile.imread(str(warped_files[0]))
    if sample.ndim == 2:
        sample = sample[:, :, None]
    n_ch = sample.shape[2]
    fig, axes = plt.subplots(1, n_ch, figsize=(4 * n_ch, 4))
    if n_ch == 1:
        axes = [axes]
    for c, ax in enumerate(axes):
        ax.imshow(sample[:, :, c], cmap="gray")
        ax.set_title(channel_dirname(c), fontsize=9)
        ax.axis("off")
    plt.suptitle(f"Warped channels: {warped_files[0].stem}", fontsize=13)
    plt.tight_layout()
    plt.show()

## Side-by-side: H&E aligned vs warped IF (channel 0)

In [ ]:
he_by_stem      = {f.stem: f for f in sorted((Path(ALIGNMENT_OUTPUT_FOLDER) / "aligned").glob("*.png"))}
seg_by_stem_out = {f.stem: f for f in sorted(out_dir.glob("*.tif"))}
common_stems    = [s for s in he_by_stem if s in seg_by_stem_out]

n_show = min(len(common_stems), 4)
if n_show == 0:
    print("No matching pairs found.")
else:
    fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
    if n_show == 1:
        axes = axes[:, np.newaxis]
    for k, stem in enumerate(common_stems[:n_show]):
        he = np.array(Image.open(str(he_by_stem[stem])))
        ifimg = tifffile.imread(str(seg_by_stem_out[stem]))
        ch0 = ifimg[:, :, 0] if ifimg.ndim == 3 else ifimg
        axes[0, k].imshow(he)
        axes[0, k].set_title(f"H&E {stem}", fontsize=8); axes[0, k].axis("off")
        axes[1, k].imshow(ch0, cmap="gray")
        axes[1, k].set_title(f"IF ch0 {stem}", fontsize=8); axes[1, k].axis("off")
    plt.suptitle("H&E aligned (top) vs warped IF ch0 (bottom)", fontsize=12)
    plt.tight_layout()
    plt.show()

    print(f"\n{'Stem':<50} {'H&E':>14} {'IF':>16} {'Match?'}")
    print("-" * 90)
    for stem in common_stems:
        hs = np.array(Image.open(str(he_by_stem[stem]))).shape[:2]
        ss = tifffile.imread(str(seg_by_stem_out[stem])).shape[:2]
        print(f"{stem:<50} {str(hs):>14} {str(ss):>16}  {'\u2713' if hs == ss else '\u2717 MISMATCH'}")